# Task 3: Energy Consumption Time Series Forecasting
# Problem Statement & Objective

* **Problem Statement:** Household energy consumption is highly volatile and depends on time-based patterns. Managing power grids requires accurate short-term forecasts to prevent outages and optimize distribution.
  
* **Objective:** To forecast short-term household energy usage by engineering time-based features and comparing the performance of **ARIMA,** **Prophet,** and **XGBoost** models.

# Dataset Description & Loading
The household power consumption dataset is imported. To facilitate time-series analysis,the 'Date' and 'Time' columns are merged into a standard datetime index.

In [ ]:
import pandas as pd

file_path = 'household_power_consumption.csv'
df = pd.read_csv(file_path, sep=';', low_memory=False, na_values=['?'])

df.columns = df.columns.str.strip()

if 'Date' in df.columns and 'Time' in df.columns:
    df['dt'] = pd.to_datetime(df['Date'] + ' ' + df['Time'], dayfirst=True, errors='coerce')
    df = df.dropna(subset=['dt'])
    df.set_index('dt', inplace=True)
    
    df['Global_active_power'] = pd.to_numeric(df['Global_active_power'], errors='coerce')
    df_hourly = df['Global_active_power'].resample('H').mean().ffill()
    
    print(" Data resampled to Hourly frequency.")
    print(df_hourly.head())
else:
    print( df.columns.tolist()) 

In [ ]:
import pandas as pd
import numpy as np

# Loading and parsing date/time
# 'household_power_consumption.csv'
df = pd.read_csv('household_power_consumption.csv', sep=';', 
                parse_dates={'datetime': ['Date', 'Time']}, 
                infer_datetime_format=True, 
                low_memory=False, index_col='datetime')

# Cleaning missing values (marked as '?' in this dataset)
df.replace('?', np.nan, inplace=True)
df = df.astype(float).dropna()

print("Step 1: Data Loaded and Cleaned Successfully!")
df.head() 

# Feature Engineering
The objective of this step is to transform the raw timestamp into meaningful numerical features that a machine learning model can process.
* **Approach:** > * Since energy consumption follows strong cyclical patterns (daily and weekly), we extract specific time components from the index.
* **Hour of Day:** Captures daily peaks, such as increased electricity usage during evening hours.
* **Day of Week:** Distinguishes between weekdays and weekends, as household activities often shift during the search.
* **Month:** Accounts for seasonal variations in energy consumption (e.g., higher usage for heating in winter or cooling in summer).
* **Is_Weekend:** A binary feature to specifically flag Saturdays and Sundays for the model.

In [ ]:
# Function to create time-series features
def create_features(df):
    df = df.copy()
    if isinstance(df, pd.Series):
        df = df.to_frame()
    
    # Extracting core time features
    df['hour'] = df.index.hour
    df['day_of_week'] = df.index.dayofweek
    df['quarter'] = df.index.quarter
    df['month'] = df.index.month
    df['year'] = df.index.year
    df['day_of_year'] = df.index.dayofyear
    
    # Identifying weekends (Energy usage often differs on weekends)
    df['is_weekend'] = df['day_of_week'].isin([5, 6]).astype(int)
    
    return df

# Applying the function to your resampled data
df_features = create_features(df_hourly)

print(df_features.head()) 

In [ ]:
# Resampling data to daily frequency
daily_df = df.resample('D').sum()

# Engineering time-based features
daily_df['hour'] = daily_df.index.hour
daily_df['day_of_week'] = daily_df.index.dayofweek
daily_df['is_weekend'] = daily_df['day_of_week'].apply(lambda x: 1 if x >= 5 else 0)

print("Feature Engineering Completed.") 

# Exploratory Data Analysis (EDA)
 The goal of EDA is to visually inspect the energy consumption data to identify trends, seasonality, and potential outliers that could affect model performance.
* **Analysis Approach:**
* **Time-Series Trend:** We plot the hourly energy consumption over time to observe long-term fluctuations and ensure the data is stationary enough for forecasting.
* **Hourly Distribution:** Using boxplots, we analyze how energy usage varies throughout the 24 hours of a day. This helps identify "Peak Hours" when household activity is at its highest.
* **Daily & Monthly Seasonality:** We examine patterns across different days of the week and months of the year to account for weekend behaviors and seasonal weather impacts (e.g., higher heating/cooling needs).
* **Key Findings:**
 . Initial visualization shows distinct cyclical patterns, confirming that energy usage is highly dependent on the time of day.
 . This confirms that Feature Engineering (extracting hour and day) will be essential for the success of our XGBoost and ARIMA models.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# 1. Clean column names (Removes hidden spaces)
df.columns = df.columns.str.strip()

# 2. Fix Date/Time Parsing (This prevents the 'Empty' graph)
try:
    # Auto-detect column names regardless of case
    date_col = [c for c in df.columns if 'date' in c.lower()][0]
    time_col = [c for c in df.columns if 'time' in c.lower()][0]
    power_col = [c for c in df.columns if 'active_power' in c.lower()][0]

    # Combine and convert to Datetime with explicit day-first format
    df['dt'] = pd.to_datetime(df[date_col].astype(str) + ' ' + df[time_col].astype(str), dayfirst=True, errors='coerce')
    
    # Drop rows that failed to parse and set as index
    df_eda = df.dropna(subset=['dt']).copy()
    df_eda.set_index('dt', inplace=True)

    # 3. Handle numeric data and resample
    df_eda[power_col] = pd.to_numeric(df_eda[power_col], errors='coerce')
    
    # Fill gaps to ensure a continuous line in the plot
    df_hourly = df_eda[power_col].resample('h').mean().ffill().to_frame(name='Power')
    df_hourly['hour'] = df_hourly.index.hour

    # --- Visualizations ---
    plt.style.use('ggplot')

    # Plot 1: Trend Line
    plt.figure(figsize=(15, 5))
    plt.plot(df_hourly.index, df_hourly['Power'], color='#2c7bb6', linewidth=1)
    plt.title('Hourly Energy Consumption Trend')
    plt.ylabel('Power (kW)')
    plt.show()

    # Plot 2: Peak Usage Boxplot
    plt.figure(figsize=(12, 6))
    sns.boxplot(data=df_hourly, x='hour', y='Power', palette='viridis')
    plt.title('Energy Usage Distribution by Hour')
    plt.ylabel('Power (kW)')
    plt.show()

    print("EDA Visualizations successfully generated.")

except Exception as e:
    print(f"Check your CSV column names. Error: {e}") 

# Model Building and Evaluation
In this step, three distinct forecasting models are implemented: **ARIMA** (Statistical), Prophet (Additive/Seasonal), and **XGBoost** (Gradient Boosting). The dataset is partitioned chronologically to ensure that the models are evaluated on "future" unseen data. Mean Absolute Error (MAE) and Root Mean Squared Error **(RMSE)** are utilized as the primary evaluation metrics to assess predictive accuracy.

In [ ]:
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error
from xgboost import XGBRegressor
from statsmodels.tsa.arima.model import ARIMA
from prophet import Prophet

# --- 1. Data Partitioning ---
# The data is split chronologically into training (80%) and testing (20%) sets
train_size = int(len(df_hourly) * 0.8)
train, test = df_hourly.iloc[:train_size], df_hourly.iloc[train_size:]

X_train, y_train = train[['hour', 'dayofweek', 'month']], train['Power']
X_test, y_test = test[['hour', 'dayofweek', 'month']], test['Power']

# --- 2. Model Training ---

# XGBoost model is initialized and fitted to the training features
xgb_model = XGBRegressor(n_estimators=1000, learning_rate=0.01)
xgb_model.fit(X_train, y_train)
xgb_preds = xgb_model.predict(X_test)

# ARIMA model is fitted to the target series
arima_model = ARIMA(y_train, order=(5,1,0))
arima_result = arima_model.fit()
arima_preds = arima_result.forecast(steps=len(test))

# Prophet model is trained using a formatted dataframe
prophet_df = train.reset_index().rename(columns={'dt': 'ds', 'Power': 'y'})
p_model = Prophet()
p_model.fit(prophet_df)
future = test.reset_index().rename(columns={'dt': 'ds'})[['ds']]
forecast = p_model.predict(future)
prophet_preds = forecast['yhat'].values

# --- 3. Performance Evaluation ---

def evaluate(y_true, y_pred, name):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    print(f"{name} -> MAE: {mae:.4f}, RMSE: {rmse:.4f}")
    return mae, rmse

print("--- Model Performance Comparison ---")
results_xgb = evaluate(y_test, xgb_preds, "XGBoost")
results_arima = evaluate(y_test, arima_preds, "ARIMA")
results_prophet = evaluate(y_test, prophet_preds, "Prophet") 

In [ ]:
# Libraries are installed to resolve ModuleNotFoundError
!pip install xgboost prophet

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error
from xgboost import XGBRegressor
from statsmodels.tsa.arima.model import ARIMA
from prophet import Prophet

# 1. Data Partitioning
# The dataset is split into 80% training and 20% testing sets chronologically
train_size = int(len(df_hourly) * 0.8)
train, test = df_hourly.iloc[:train_size], df_hourly.iloc[train_size:]

X_train, y_train = train[['hour', 'dayofweek', 'month']], train['Power']
X_test, y_test = test[['hour', 'dayofweek', 'month']], test['Power']

# 2. Model Implementation
# --- XGBoost ---
xgb_model = XGBRegressor(n_estimators=100, learning_rate=0.05)
xgb_model.fit(X_train, y_train)
xgb_preds = xgb_model.predict(X_test)

# --- ARIMA ---
# Frequency is set to 'H' (Hourly) to avoid ARIMA warnings
y_train_arima = y_train.copy()
y_train_arima.index.freq = 'H' 
arima_model = ARIMA(y_train_arima, order=(5,1,0))
arima_result = arima_model.fit()
arima_preds = arima_result.forecast(steps=len(test))

# --- Prophet ---
prophet_train = train.reset_index().rename(columns={'dt': 'ds', 'Power': 'y'})
p_model = Prophet()
p_model.fit(prophet_train)
future = test.reset_index().rename(columns={'dt': 'ds'})[['ds']]
forecast = p_model.predict(future)
prophet_preds = forecast['yhat'].values

# 3. Performance Evaluation Function
# Errors like 'unterminated string' are fixed here by closing all quotes
def evaluate_performance(y_true, y_pred, model_name):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    print(f"{model_name} -> MAE: {mae:.4f}, RMSE: {rmse:.4f}")
    return mae, rmse

print("--- Final Model Comparison ---")
results_xgb = evaluate_performance(y_test, xgb_preds, "XGBoost")
results_arima = evaluate_performance(y_test, arima_preds, "ARIMA")
results_prophet = evaluate_performance(y_test, prophet_preds, "Prophet")

# 4. Results Visualization
plt.figure(figsize=(15, 5))
plt.plot(y_test.index, y_test.values, label='Actual', color='black', alpha=0.3)
plt.plot(y_test.index, xgb_preds, label='XGBoost', color='red')
plt.title('Energy Consumption Forecast Comparison')
plt.legend()
plt.show() 